# Audiovisual Cortical Mapping Pipeline

Runs the three-script pipeline sequentially:

| Script | What it does | Key outputs |
|--------|-------------|-------------|
| `01_extract_geometry.py` | Glasser → Audio/Video masks → sphere LBOEs | `sub_audio.pkl`, `sub_vidoe.pkl` |
| `02_prep_hcp_timeseries.py` | MAT → z-score → project onto LBOEs | `X_train/test.npy`, `Y_train/test.npy` |
| `03_fit_banded_ridge.py` | Himalaya banded ridge → variance partition | `R2_*.npy`, `*.dscalar.nii` |

**Kernel:** `vicsompy_av`  
**Estimated wall-clock:** ~2 h (dominated by Script 1 LBOE computation and Script 3 fitting)

In [20]:
import importlib.util
import sys
import os
import time
import logging
import numpy as np

ROI_DEFS = {
    "A1": {"L": 204, "R": 24},   
    "V1": {"L": 181, "R": 2}, 
    "TA2": {"L": 287, "R": 107},   
    "MST": {"L": 182, "R": 2},     
    "A5": {"L": 305, "R": 125},   
    "FFC": {"L": 198, "R": 18},}

os.environ["AUDIO_ROI"] = "A1"
os.environ["VIDEO_ROI"] = "V1"

# ── point Python at the scripts directory ─────────────────────────────────────
SCRIPT_DIR = os.path.dirname(os.path.abspath("run_pipeline.ipynb"))
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

# ── configure logging so it shows up in notebook cells ────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,           # re-configure if already set by an imported module
    handlers=[logging.StreamHandler(sys.stdout)],
)

def load_script(name):
    """Import a pipeline script as a module by file path."""
    path = os.path.join(SCRIPT_DIR, name)
    spec = importlib.util.spec_from_file_location(name.replace(".py", ""), path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def banner(title):
    bar = "═" * 60
    print(f"\n{bar}")
    print(f"  {title}")
    print(bar)

print("Notebook initialised.")
print(f"  Script dir : {SCRIPT_DIR}")
print(f"  Python     : {sys.version.split()[0]}")

Notebook initialised.
  Script dir : /home/amin/Research/Representation/Movie/movie_watching/vicsompy_audiovisual
  Python     : 3.10.20


---
## Script 1 — Extract Geometry & LBOEs

- Loads Glasser GIFTI parcellations → boolean masks for bilateral A1 and V1
- Loads HCP sphere surfaces directly via nibabel (bypasses pycortex DB)
- Runs Laplace-Beltrami decomposition → 200 eigenfunctions per hemisphere
- Caches results as `sub_a1.pkl` / `sub_v1.pkl` (re-run loads from cache)

> **Note:** First run takes ~5–15 min per ROI (sparse eigendecomposition). Subsequent runs load from cache instantly.

In [21]:
banner("SCRIPT 1 — Extract Geometry & LBOEs")
t0 = time.time()

s1 = load_script("01_extract_geometry.py")
s1.main()

elapsed = time.time() - t0
print(f"\n✓ Script 1 complete in {elapsed/60:.1f} min")


════════════════════════════════════════════════════════════
  SCRIPT 1 — Extract Geometry & LBOEs
════════════════════════════════════════════════════════════
12:33:38  INFO  ============================================================
12:33:38  INFO  Script 1 — Extract geometry & LBOEs
12:33:38  INFO    Audio ROI        : A1
12:33:38  INFO    Video ROI        : V1
12:33:38  INFO    Pycortex subject : hcp_999999_draw_NH
12:33:38  INFO    Surface type     : sphere (HCP GIFTI, vicsompy default)
12:33:38  INFO    LBOEs per hem.   : 200
12:33:38  INFO  ============================================================
12:33:38  INFO  Loading Glasser GIFTI parcellations …
12:33:38  INFO    Left : 32492 vertices, 181 unique labels
12:33:38  INFO    Right: 32492 vertices, 181 unique labels
12:33:38  INFO    A1 Left: 77 vertices
12:33:38  INFO    A1 Right: 66 vertices
12:33:38  INFO    V1 Left: 831 vertices
12:33:38  INFO    V1 Right: 76 vertices
12:33:38  INFO  
Building A1 subsurface …
12:33:38 

In [22]:
# ── Quick sanity check ────────────────────────────────────────────────────────
import pickle, os

CACHE_DIR = s1.CACHE_DIR
for roi in [s1.AUDIO_ROI, s1.VIDEO_ROI]:
    pkl = os.path.join(CACHE_DIR, f"sub_{roi.lower()}.pkl")
    with open(pkl, "rb") as fh:
        sub = pickle.load(fh)
    # Backward-compat: old pickles may lack n_lboe attribute
    if not hasattr(sub, "n_lboe"):
        sub.n_lboe = sub.L_eigenvectors.shape[1]
    n_lboe = sub.n_lboe
    print(f"sub_{roi.lower()}:")
    print(f"  subsurface_verts_L : {len(sub.subsurface_verts_L)} verts   "
          f"(indices in [0, 32491])")
    print(f"  subsurface_verts_R : {len(sub.subsurface_verts_R)} verts   "
          f"(indices in [32492, 64983])")
    print(f"  L_eigenvectors     : {sub.L_eigenvectors.shape}  dtype={sub.L_eigenvectors.dtype}")
    print(f"  R_eigenvectors     : {sub.R_eigenvectors.shape}")
    print(f"  n_lboe (actual)    : {n_lboe}  (target max: {s1.N_LBOE})")
    # Both hemispheres must have the same eigenfunction count
    assert sub.L_eigenvectors.shape[1] == sub.R_eigenvectors.shape[1], \
        "L/R LBOE count mismatch within ROI!"
    # Actual count must not exceed the configured target
    assert n_lboe <= s1.N_LBOE, \
        f"n_lboe={n_lboe} exceeds N_LBOE target={s1.N_LBOE}!"
    # Right hemisphere vertex indices must carry the +32492 offset
    assert sub.subsurface_verts_R.min() >= 32492, "R verts offset missing!"
    print(f"  ✓ assertions passed\n")

sub_a1:
  subsurface_verts_L : 77 verts   (indices in [0, 32491])
  subsurface_verts_R : 66 verts   (indices in [32492, 64983])
  L_eigenvectors     : (77, 64)  dtype=complex128
  R_eigenvectors     : (66, 64)
  n_lboe (actual)    : 64  (target max: 200)
  ✓ assertions passed

sub_v1:
  subsurface_verts_L : 831 verts   (indices in [0, 32491])
  subsurface_verts_R : 76 verts   (indices in [32492, 64983])
  L_eigenvectors     : (831, 74)  dtype=complex128
  R_eigenvectors     : (76, 74)
  n_lboe (actual)    : 74  (target max: 200)
  ✓ assertions passed



---
## Script 2 — Prepare HCP Timeseries

- Loads the two hemisphere MAT files (~785 MB each)
- Parses `movie_timing.csv` → correct MAT offsets for each video segment
- Applies vicsompy-style train/test split:
  - **Train:** videos 1–4, 6–8, 10–13, 15–17 — z-scored per-run
  - **Test:** videos 5, 9, 14, 18 (82 TRs each) — z-scored per clip, concatenated → 328 TRs
- Projects BOLD onto LBOEs → design matrix X (T × 800)
- Extracts CIFTI grayordinate targets Y (T × 59412)

In [23]:
banner("SCRIPT 2 — Prepare HCP Timeseries")
t0 = time.time()

s2 = load_script("02_prep_hcp_timeseries.py")
s2.main()

elapsed = time.time() - t0
print(f"\n✓ Script 2 complete in {elapsed/60:.1f} min")


════════════════════════════════════════════════════════════
  SCRIPT 2 — Prepare HCP Timeseries
════════════════════════════════════════════════════════════
12:33:49  INFO  ============================================================
12:33:49  INFO  Script 2 — Prepare HCP timeseries
12:33:49  INFO  ============================================================
12:33:49  INFO  
Loading subsurfaces from Script 1 …
12:33:49  INFO    A1 — L eigenvectors: (77, 64), R eigenvectors: (66, 64)  (n_lboe=64)
12:33:49  INFO    V1 — L eigenvectors: (831, 74), R eigenvectors: (76, 74)  (n_lboe=74)
12:33:49  INFO  
Loading fMRI MAT files …
12:33:49  INFO    Loading notmean_left_Meanfile.mat …
12:33:52  INFO      Found key='Left_data', shape=(32492, 3655)
12:33:52  INFO      Final shape: (32492, 3655)  (verts × time)
12:33:52  INFO    Loading notmean_right_Meanfile.mat …
12:33:55  INFO      Found key='Right_data', shape=(32492, 3655)
12:33:55  INFO      Final shape: (32492, 3655)  (verts × time)
12:33

In [24]:
# ── Verify output shapes ──────────────────────────────────────────────────────
import pickle

PREP_DIR  = s2.PREP_DIR
CACHE_DIR = s2.CACHE_DIR

# Determine actual design-matrix column counts from the cached subsurfaces
# (n_lboe may be < 200 for small ROIs like A1)
with open(os.path.join(CACHE_DIR, f"sub_{s2.AUDIO_ROI.lower()}.pkl"), "rb") as fh:
    _sub_a1 = pickle.load(fh)
with open(os.path.join(CACHE_DIR, f"sub_{s2.VIDEO_ROI.lower()}.pkl"), "rb") as fh:
    _sub_v1 = pickle.load(fh)
n_audio_cols = _sub_a1.L_eigenvectors.shape[1] + _sub_a1.R_eigenvectors.shape[1]
n_video_cols = _sub_v1.L_eigenvectors.shape[1] + _sub_v1.R_eigenvectors.shape[1]
expected_x_cols = n_audio_cols + n_video_cols
print(f"Expected X cols: {expected_x_cols}  "
      f"(audio={n_audio_cols}, video={n_video_cols})")

arrays = {}
expected_shapes = {
    "X_train"   : (None, expected_x_cols),
    "Y_train"   : (None, 59412),
    "X_test"    : (328,  expected_x_cols),
    "Y_test"    : (328,  59412),
    "run_onsets": (4,),
}

print("\nLoaded arrays:")
for name, exp in expected_shapes.items():
    arr = np.load(os.path.join(PREP_DIR, f"{name}.npy"))
    arrays[name] = arr
    ok = all(e is None or arr.shape[i] == e for i, e in enumerate(exp))
    status = "✓" if ok else "✗ MISMATCH"
    print(f"  {name:12s}: {str(arr.shape):25s}  expected {str(exp):25s}  {status}")

band_sizes = np.load(os.path.join(PREP_DIR, "band_sizes.npy"))
print(f"\nband_sizes      : audio={band_sizes[0]}, video={band_sizes[1]}")
print(f"Run onsets      : {arrays['run_onsets'].tolist()}")
print(f"T_train         : {arrays['X_train'].shape[0]} TRs")
print(f"T_test          : {arrays['X_test'].shape[0]} TRs  (4 × 82)")
print(f"Design matrix   : audio cols 0-{n_audio_cols-1}, video cols {n_audio_cols}-{expected_x_cols-1}")

# Basic value sanity: z-scored data should have near-zero mean
print(f"\nY_train mean    : {arrays['Y_train'].mean():.4f}  (expect ≈0)")
print(f"Y_train std     : {arrays['Y_train'].std():.4f}")
print(f"X_train has NaN : {np.isnan(arrays['X_train']).any()}  (expect False)")
print(f"Y_train has NaN : {np.isnan(arrays['Y_train']).any()}  (expect False)")

Expected X cols: 276  (audio=128, video=148)

Loaded arrays:
  X_train     : (2834, 276)                expected (None, 276)                ✓
  Y_train     : (2834, 59412)              expected (None, 59412)              ✓
  X_test      : (328, 276)                 expected (328, 276)                 ✓
  Y_test      : (328, 59412)               expected (328, 59412)               ✓
  run_onsets  : (4,)                       expected (4,)                       ✓

band_sizes      : audio=128, video=148
Run onsets      : [0, 680, 1412, 2118]
T_train         : 2834 TRs
T_test          : 328 TRs  (4 × 82)
Design matrix   : audio cols 0-127, video cols 128-275

Y_train mean    : -0.0000  (expect ≈0)
Y_train std     : 1.0000
X_train has NaN : False  (expect False)
Y_train has NaN : False  (expect False)


---
## Script 3 — Fit Banded Ridge Regression

- Sets himalaya torch backend
- Builds `ColumnKernelizer` → `MultipleKernelRidgeCV` pipeline
  - Audio band: `slice(0, 400)` — bilateral A1 LBOEs
  - Video band: `slice(400, 800)` — bilateral V1 LBOEs
- Fits with leave-one-run-out CV (4 folds, `random_search`, 20 iterations)
- Evaluates on held-out 328-TR test set
- Computes variance partitioning:
  - `R2_audio`, `R2_video` — unique variance per modality
  - `Shared_R2 = R2_audio + R2_video − R2_full` — audiovisual integration


In [25]:
banner("SCRIPT 3 — Fit Banded Ridge Regression")
t0 = time.time()

s3 = load_script("03_fit_banded_ridge.py")
s3.main()

elapsed = time.time() - t0
print(f"\n✓ Script 3 complete in {elapsed/60:.1f} min")


════════════════════════════════════════════════════════════
  SCRIPT 3 — Fit Banded Ridge Regression
════════════════════════════════════════════════════════════
12:33:57  INFO  ============================================================
12:33:57  INFO  Script 3 — Fit banded ridge regression
12:33:57  INFO    Backend       : torch  (vicsompy default)
12:33:57  INFO    Solver        : random_search  (vicsompy default)
12:33:57  INFO    Alphas        : logspace(1,20,20)
12:33:57  INFO    n_iter        : 20
12:33:57  INFO    Targets batch : 400
12:33:57  INFO  ============================================================
12:33:57  INFO  
Initialising himalaya backend: 'torch' …
12:33:57  INFO    Backend ready: <module 'himalaya.backend.torch' from '/home/amin/miniconda3/envs/vicsompy_av/lib/python3.10/site-packages/himalaya/backend/torch.py'>
12:33:57  INFO  Loading pre-processed arrays from /home/amin/Research/Representation/Movie/outputs/vicsompy_audiovisual/A1_V1/prep …
12:33:57  INF

In [26]:
s4 = load_script("04_null_corrected_maps.py")
s4.main()
s5 = load_script("05_audiovisual_integration_maps.py")
s5.main()
# s6 = load_script("06_rsa_overlap.py")
# s6.main()

12:39:18  INFO  ============================================================
12:39:18  INFO  Script 04 — Null-corrected R² maps (Hedger et al. Fig 3a method)
12:39:18  INFO  ============================================================
12:39:18  INFO  
Loading subsurface pickles …
12:39:18  INFO    A1: n_lboe=64  L=(77, 64)  R=(66, 64)
12:39:18  INFO    V1: n_lboe=74  L=(831, 74)  R=(76, 74)
12:39:18  INFO  
Loading test arrays …
12:39:18  INFO    X_test: (328, 276)   Y_test: (328, 59412)
12:39:18  INFO  
Loading split R² from Script 03 …
12:39:18  INFO    R2_A1: mean=0.0706  frac>0=85.6%
12:39:18  INFO    R2_V1: mean=0.3376  frac>0=98.4%
12:39:18  INFO  
Reconstructing mean A1/V1 time courses from LBOE projections …
12:39:18  INFO    mean_A1: shape=(328,)  std=0.9974
12:39:18  INFO    mean_V1: shape=(328,)  std=0.5657
12:39:18  INFO  
Fitting null model for audio (mean A1 regressor) …
12:39:19  INFO  Fitting null model for video (mean V1 regressor) …
12:39:19  INFO    R2_null_A1: mean=